# Flight Delays - First Attempt

Quick model to predict whether a US domestic flight will arrive **15 or more minutes late** (arrival delay flag `ArrDel15`).

Who wrote this: me, days ago, in a hurry. The goal of the whole course is to turn this notebook into something that can be operated.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
# load the raw on-time performance data from the course data pool
df = pd.read_csv("../../../../data/flight_delays_2025_01.csv")
df.head()

,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Flight_Number_Reporting_Airline,Origin,OriginCityName,...,Diverted,CRSElapsedTime,ActualElapsedTime,AirTime,Distance,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2025,1,1,1,3,2025-01-01,AA,1,JFK,"New York, NY",...,0.0,381.0,377.0,345.0,2475.0,NaN,NaN,NaN,NaN,NaN
1,2025,1,1,2,4,2025-01-02,AA,1,JFK,"New York, NY",...,0.0,381.0,390.0,353.0,2475.0,NaN,NaN,NaN,NaN,NaN
2,2025,1,1,3,5,2025-01-03,AA,1,JFK,"New York, NY",...,0.0,381.0,371.0,347.0,2475.0,NaN,NaN,NaN,NaN,NaN
3,2025,1,1,4,6,2025-01-04,AA,1,JFK,"New York, NY",...,0.0,386.0,383.0,349.0,2475.0,NaN,NaN,NaN,NaN,NaN
4,2025,1,1,5,7,2025-01-05,AA,1,JFK,"New York, NY",...,0.0,381.0,378.0,347.0,2475.0,NaN,NaN,NaN,NaN,NaN


In [3]:
df.shape

(539747, 40)

In [4]:
# how often is a flight late (>= 15 min)?
df["ArrDel15"].value_counts(normalize=True)

ArrDel15
0.0    0.812108
1.0    0.187892
Name: proportion, dtype: float64

## Clean up

Drop cancelled and diverted flights - they are not really delays - and remove rows with a missing target.

In [5]:
df = df[df["Cancelled"] == 0]
df = df[df["Diverted"] == 0]
df = df.dropna(subset=["ArrDel15"])
df = df.fillna(0)
print("rows after cleaning:", len(df))

rows after cleaning: 522269


C:\Users\A7mad\AppData\Local\Temp\ipykernel_19808\249810641.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(0)


In [6]:
# belt and suspenders: make sure no cancelled/diverted rows survive
df = df[df["Cancelled"] == 0]
df = df[df["Diverted"] == 0]
df = df.drop_duplicates()
print("rows after extra pass:", len(df))

rows after extra pass: 522269


## Feature engineering

Departure delay and flight distance are the obvious signals. Keep it simple.

In [7]:
# hour of the scheduled departure
df["hour"] = df["CRSDepTime"] // 100

# cap extreme departure delays at 3 hours so outliers don't dominate
def cap(v):
    return min(v, 180)

df["depdelay_cap"] = df["DepDelay"].apply(cap)

In [8]:
# one-hot encode the carriers and airports
cat = pd.get_dummies(df[["Reporting_Airline", "Origin", "Dest"]],
                     columns=["Reporting_Airline", "Origin", "Dest"])
num_cols = ["depdelay_cap", "Distance", "CRSElapsedTime", "hour", "DayOfWeek"]
num = df[num_cols].copy()
num["Distance"] = num["Distance"].fillna(num["Distance"].median())

print(num.head())

   depdelay_cap  Distance  CRSElapsedTime  hour  DayOfWeek
0          -3.0    2475.0           381.0     6          3
1          -7.0    2475.0           381.0     6          4
2          -7.0    2475.0           381.0     6          5
3          -7.0    2475.0           386.0     7          6
4          -4.0    2475.0           381.0     6          7


In [9]:
# scale the numeric columns so distances don't dwarf the rest
sc = StandardScaler()
sc_num = sc.fit_transform(num)
sc_num = pd.DataFrame(sc_num, columns=num_cols)

X = pd.concat([cat.reset_index(drop=True), sc_num.reset_index(drop=True)], axis=1)
y = df["ArrDel15"].values
print("X shape:", X.shape)

X shape: (522269, 677)


## Train

Split, fit a random forest, done.

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [11]:
y_pred = model.predict(X_test)
print("accuracy:", accuracy_score(y_test, y_pred))
print("f1:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

accuracy: 0.9183947000593563
f1: 0.7587319558448911
              precision    recall  f1-score   support

         0.0       0.93      0.97      0.95     84941
         1.0       0.85      0.69      0.76     19513

    accuracy                           0.92    104454
   macro avg       0.89      0.83      0.85    104454
weighted avg       0.92      0.92      0.91    104454



In [12]:
import joblib
os.makedirs("models", exist_ok=True)
joblib.dump(model, "models/model_2025_01.joblib")
print("saved model to models/model_2025_01.joblib")

saved model to models/model_2025_01.joblib


In [13]:
# scratch - how many rows per airline?
df.groupby("Reporting_Airline").size().sort_values(ascending=False)

Reporting_Airline
WN    102120
DL     74025
AA     72082
OO     63502
UA     60668
YX     26683
MQ     20831
OH     19151
AS     17847
B6     17558
NK     16946
F9     15110
G4      9206
HA      6540
dtype: int64

## Notes to self

- Baseline accuracy/f1 captured above. This is the number to beat.
- Will clean this up later, once everything else is working.